# NV-Diamond Quantum Spin Dynamics — Student Notebook
**Student:** [Name]  
**Period:** [Period]  
**Date:** [Date]  
**Environment:** [Colab / OnDemand Jupyter / VS Code]

Complete Parts 0–3 in sequence using the GitHub instructions. Figures 1–3 have **layout-only guides** in the Markdown; Figure 4 has **no guide** because you must independently design its standard scientific comparison graph.

**Prompting progression:** Part 0 gives prompting principles; Part 1 Figure 1 provides the **only complete example LLM prompt**. Construct your own prompts for every later coding task. Do not use AI to write your physics-analysis answers.

**Submit one PDF** of this completed notebook, including all computations, Figures 1–4, captions, numerical tables, predictions, analysis, AI records, and final Submission Check.

## Part 0 — Setup and Analytical Warm-Up
### CODE — PROVIDED · Environment and basis

In [ ]:
# Run this cell first. If QuTiP is missing, install it in a separate cell:
# %pip install qutip numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from matplotlib.figure import Figure
from matplotlib.patches import Patch

zero, one = qt.basis(2, 0), qt.basis(2, 1)
psi_plus = (zero + one).unit()
sx, sy, sz = qt.sigmax(), qt.sigmay(), qt.sigmaz()
projector0, projector1 = zero * zero.dag(), one * one.dag()
figures = {}               # Register Figures 1–4 after creating them.
print(f"QuTiP {qt.__version__} — environment ready")

### CODE — PROVIDED · Shared simulation and plotting utilities (run once)
**Axis overlays:** `plot_bloch(..., omega_z=..., omega_x=..., axis_unit=..., axis_scale=...)` draws the effective Hamiltonian components. These are rotation terms, **not** laboratory field vectors. For Figure 3 use the **same** `axis_unit=omega1` and `axis_scale=0.28` on all four spheres; this makes arrow lengths directly comparable.


In [ ]:
def simulate_spin(H, initial_state, times):
    """Numerical H is H/hbar (rad/us); times are in us.

    ALL time-evolution simulations return the same fields:
    times, states, p0, p1, bloch.
    """
    times = np.asarray(times, dtype=float)
    if times.ndim != 1 or len(times) < 2 or np.any(np.diff(times) <= 0):
        raise ValueError("times must be a strictly increasing array of at least two values")
    solution = qt.sesolve(H, initial_state, times)
    states = solution.states
    if len(states) != len(times):
        raise RuntimeError("Solver did not return states at every time point")
    return {
        "times": times,
        "states": states,
        "p0": np.asarray(qt.expect(projector0, states), dtype=float),
        "p1": np.asarray(qt.expect(projector1, states), dtype=float),
        "bloch": np.asarray([qt.expect(op, states) for op in (sx, sy, sz)], dtype=float),
    }


def fluorescence(p0, p1, F0=100, F1=70):
    """Expected photons in a fixed readout window."""
    return F0 * np.asarray(p0) + F1 * np.asarray(p1)


def plot_bloch(ax, simulation, title="", color="tab:blue", *,
               omega_z=None, omega_x=None, axis_unit=None, axis_scale=0.90):
    """Draw a Bloch trajectory and its Hamiltonian rotation components.

    omega_z: coefficient of sigma_z (rad/us), labeled omega in Part 1
             or -Delta_omega in Part 2, depending on the calling cell.
    omega_x: coefficient of sigma_x (rad/us), labeled omega_1.
    axis_unit: common angular-frequency unit used to scale component arrows;
               hold constant across all four detunings for honest comparison.
    axis_scale: Bloch-coordinate arrow length per axis_unit.

    Arrows represent EFFECTIVE ROTATION TERMS in the Hamiltonian, not physical
    laboratory magnetic/microwave field vectors. Zero components are omitted.
    """
    u = np.linspace(0, 2*np.pi, 35)
    v = np.linspace(0, np.pi, 19)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(x, y, z, color="0.83", linewidth=0.35, alpha=0.45)
    for axis, label in [(0, 'x'), (1, 'y'), (2, 'z')]:
        vec = np.zeros(3)
        vec[axis] = 1.16
        ax.plot([-vec[0], vec[0]], [-vec[1], vec[1]],
                [-vec[2], vec[2]], color='0.58', linewidth=0.7)
        ax.text(*vec, label)

    xyz = np.asarray(simulation['bloch'])
    ax.plot(*xyz, color=color, linewidth=2)
    ax.scatter(*xyz[:, 0], color='green', s=38, depthshade=False)
    ax.scatter(*xyz[:, -1], color=color, s=28, depthshade=False)
    direction_at = max(0, min(len(simulation['times'])//8,
                              len(simulation['times']) - 2))
    delta = xyz[:, direction_at + 1] - xyz[:, direction_at]
    if np.linalg.norm(delta) > 1e-8:
        ax.quiver(*xyz[:, direction_at], *delta, color=color,
                  arrow_length_ratio=0.5, length=1, normalize=False)

    # Show the separate z and x terms rather than a combined effective axis.
    # Lengths are proportional to their absolute angular-frequency values.
    if omega_z is not None or omega_x is not None:
        components = [(omega_z, 2, 'tab:orange', r'$\omega$' if omega_x is None
                       else r'$-\Delta\omega$'),
                      (omega_x, 0, 'tab:purple', r'$\omega_1$')]
        active = [abs(float(w)) for w, _, _, _ in components if w is not None and abs(w) > 1e-12]
        unit = float(axis_unit) if axis_unit is not None else (max(active) if active else 1.0)
        if unit <= 0:
            raise ValueError('axis_unit must be positive')
        for value, coordinate, arrow_color, label in components:
            if value is None or np.isclose(value, 0):
                continue
            arrow = np.zeros(3)
            arrow[coordinate] = axis_scale * float(value) / unit
            ax.quiver(0, 0, 0, *arrow, color=arrow_color, linewidth=3,
                      arrow_length_ratio=0.17, normalize=False)
            tip = arrow.copy()
            tip[coordinate] += 0.10 * np.sign(arrow[coordinate])
            ax.text(*tip, label, color=arrow_color, fontsize=11, weight='bold')

    ax.set(xlim=(-1.47, 1.47), ylim=(-1.47, 1.47), zlim=(-1.47, 1.47))
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=22, azim=42)
    ax.set_axis_off()
    # Fixed-position color key keeps the short component readable in print.
    if omega_z is not None:
        z_label = r'$\omega$ (z)' if omega_x is None else r'$-\Delta\omega$ (z)'
        if np.isclose(omega_z, 0):
            z_label += ' = 0'
        ax.text2D(0.02, 0.07, z_label, transform=ax.transAxes,
                  color='tab:orange', fontsize=9, weight='bold')
    if omega_x is not None:
        ax.text2D(0.02, 0.015, r'$\omega_1$ (x)', transform=ax.transAxes,
                  color='tab:purple', fontsize=9, weight='bold')
    ax.set_title(title, fontsize=10)
    return ax


def check_normalization(sim):
    """Return maximum state-norm and population-sum errors."""
    norm_error = max(abs(state.norm() - 1) for state in sim['states'])
    pop_error = np.max(np.abs(sim['p0'] + sim['p1'] - 1))
    return float(norm_error), float(pop_error)

print("Shared simulation and plotting functions ready.")

### DO NOW — Analytical Warm-Up · Work independently
Initial state: $|\psi(0)\rangle=(|0\rangle+|1\rangle)/\sqrt2$; $\hat H=\hbar\omega\sigma_z/2$.

**Energy eigenvalues:** [Your calculation]

**Time-dependent state:** [Your calculation]

**Populations $P_0(t)$ and $P_1(t)$:** [Your calculation]

**Relative phase and predicted Bloch trajectory:** [Your response]

**Warm-up output target:** Print a four-row table for 0, 0.25, 0.50 and 1.00 µs with time, `P0`, `P1`, and relative phase in radians. Compare relative phase rather than absolute complex-amplitude phase (global phase is arbitrary). No figure required. Construct your own AI prompt; specify the inputs, existing solver and required columns.

### CODE — TODO · Part 0 numerical verification
Use `omega_warm = 2*np.pi`, `times_warm` (101 points, 0–1 µs), and `simulate_spin()`.

**Required output:** Exactly four time rows (0, 0.25, 0.50, 1.00 µs) with columns **Time (µs), P0, P1, Relative phase (rad)**. Show probabilities to at least three decimals; compare the table to your hand derivation. No graph.

**Prompting practice:** Independently explain the physical model, existing variables/functions, the requested table and numerical checks to your LLM. Do not request a complete replacement notebook.

In [ ]:
# TODO: Construct H_warm = (omega_warm/2)*sz (numerical H/hbar).
# Store warmup = simulate_spin(H_warm, psi_plus, times_warm).
# Required output: print exactly 4 sampled rows at t = 0, 0.25, 0.50, 1.00 us.
# Columns: Time (us), P0, P1, relative phase in radians.
# Compute relative phase from complex amplitudes <0|psi> and <1|psi>
# (arg(amplitude_1) - arg(amplitude_0)); choose one wrapping convention.
# Check P0 + P1 approximately 1. No plot or separate solver.
omega_warm = 2*np.pi
times_warm = np.linspace(0, 1, 101)
H_warm = None                 # TODO
warmup = None                 # TODO: standard simulation-result dictionary
# TODO: print the required 4-row table and numerical normalization check.

**RESULT — Warm-up numerical agreement:** [Brief comparison with your analytical prediction]

## Part 1 — Larmor Precession and Quantum Phase
### DO NOW — Predictions (before running code)
**1. Initial Bloch location:** [Answer]  
**2. Trajectory:** [Answer]  
**3. Behavior of $P_0,P_1$:** [Answer]  
**4. Effect of doubling $\omega$:** [Answer]

### CODE — TODO · Larmor simulation (data for Figure 1)
**Parameters:** `omega = 2*np.pi` rad/µs; 501 samples over 0–2 µs; initial state `psi_plus`.

**Required output:** Store the standard result dictionary as `part1` (fields `times`, `states`, `p0`, `p1`, `bloch`). **Do not plot yet.** Figure 1 is created in the following cell.

In [ ]:
omega = 2*np.pi                         # rad/us
times1 = np.linspace(0, 2, 501)         # us
H1 = None                               # TODO: H/hbar = omega*sz/2
part1 = None                            # TODO: simulate_spin(H1, psi_plus, times1)

### CODE — TODO · Figure 1 (consult Part 1, Step 3 and `Images/figure1_layout_guide.png`)

**One Matplotlib figure with two side-by-side panels.** LEFT: Bloch trajectory from `part1`, initial state/time arrow, and labeled +z rotation term (`omega_z=omega`, `axis_unit=omega`). RIGHT: `part1["p0"]`, `part1["p1"]` versus time, with axes **Time (µs), 0–2** and **Probability, 0–1**, legend and titles. Add a 1–2 sentence caption and register `figures["Figure 1"] = fig1`.

**Only fully written LLM prompt:** The Part 1 GitHub instructions provide one worked prompt for this figure. Use it as a model when drafting your own prompts for all later tasks.

In [ ]:
# TODO: Figure 1, ONE figure with TWO side-by-side panels.
# Panel A: 3D Bloch sphere using plot_bloch(ax, part1, ...,
#                        omega_z=omega, axis_unit=omega).
# Panel B: plot part1['p0'] and part1['p1'] vs part1['times'].
# Axes: Time (us) 0-2; Probability 0-1; include title and legend.
# Mark initial state/time direction (the helper handles these).
# Save figure in fig1 and register figures['Figure 1'] = fig1.

**FIGURE 1 CAPTION —** [1–2 sentences connecting precession and unchanged populations]

### CODE — TODO · Double precession frequency: comparison table
Use `omega_fast = 4*np.pi` rad/µs, same `psi_plus` and time samples, and save `part1_fast`.

**Required output:** Two-row table with **ω (rad/µs), predicted period (µs), revolutions in 2 µs, maximum population variation**. Print max normalization error for each case. No further numbered figure. Construct your own LLM prompt from these specifications.

In [ ]:
omega_fast = 4*np.pi
H1_fast = None                       # TODO: H/hbar using sz
part1_fast = None                   # TODO: simulate_spin(..., psi_plus, times1)

# TODO: PRINT one 2-row table with columns:
# omega (rad/us) | predicted T=2*pi/abs(omega) (us) |
# revolutions in 2 us | max absolute variation of P0 or P1
# Also report max abs(P0+P1-1) for each simulation.
# No extra figure. Keep the supplied shared solver.

### DO NOW — Independent Physics Analysis
**Q1 — Time Evolution Without Population Transfer.** Why does the Bloch vector move while $P_0,P_1$ remain constant? Distinguish relative phase from basis-state probabilities.

[Type your answer here]

**Q2 — Magnetic Fields and Precession.** Why does an energy difference change relative-phase accumulation, and how does that connect to the NV spin and a magnetic field?

[Type your answer here]

## Part 2A — Resonant Microwave Driving

**Driven-system preview:** The NV spin's natural angular frequency is `ω0`, the microwave drive frequency is `ω`, and **detuning** is `Δω = ω − ω0`. The microwave is physically time dependent; the effective Hamiltonian supplied here approximates the slow spin dynamics in a rotating frame:

```math
\hat H_{\mathrm{eff}}/\hbar=(-\Delta\omega\,\sigma_z+\omega_1\sigma_x)/2.
```

The rotating-frame derivation will come later in class. Here, `omega1` denotes microwave coupling in rad/µs.

### DO NOW — Resonant predictions (complete before code)
**1. Initial Bloch location:** [Answer]  
**2. Rotation axis:** [Answer]  
**3. Population behavior:** [Answer]  
**4. First π-pulse duration at ω1=2π rad/µs:** [Answer]  
**5. Effect of doubling coupling:** [Answer]

### CODE — TODO · Define the effective NV Hamiltonian ONCE for Parts 2 and 3


In [ ]:
def nv_hamiltonian(detuning, omega1):
    """Return effective H/hbar in rad/us.

    detuning = Delta_omega = omega_drive - omega_nv (rad/us)
    omega1 = microwave coupling strength (rad/us)
    The effective z coefficient is -detuning.
    """
    # TODO: Return (-detuning*sz + omega1*sx)/2 using QuTiP operators.
    raise NotImplementedError("Implement nv_hamiltonian once; reuse it in Parts 2–3")


### CODE — TODO · Resonant Rabi simulation (data for Figure 2)
Initialize `zero`; use `omega1 = 2*np.pi`, `detuning = 0`, `times2` with 1,001 points over 0–2 µs. Use `nv_hamiltonian()` and `simulate_spin()`; store `part2_res`. Determine theoretical and observed first π-pulse durations. **Do not plot until Figure 2 cell.**

In [ ]:
omega1 = 2*np.pi
# At resonance: omega_drive = omega_nv, so Delta_omega = 0.
detuning = 0.0
times2 = np.linspace(0, 2, 1001)       # us; resolves strong-detuning oscillations
H_res = nv_hamiltonian(detuning, omega1)
part2_res = None                       # TODO: simulate_spin(H_res, zero, times2)


### CODE — TODO · Figure 2 (consult Part 2, Step 3 and `Images/figure2_layout_guide.png`)

**One figure, two side-by-side panels.** LEFT: Bloch sphere from `part2_res`, starting at `|0⟩`, time direction, labeled +x `omega1` component (`omega_x=omega1`, `axis_unit=omega1`). RIGHT: both populations versus 0–2 µs (probability 0–1), with a **labeled dashed vertical line at the first π-pulse time**. Include titles, legend, caption; register `figures["Figure 2"] = fig2`.

**Prompt construction:** Write your own LLM prompt based on the example for Figure 1. Identify the existing data, helper, layout, π-pulse marker, and output registration.

In [ ]:
# TODO: Figure 2: ONE figure with TWO side-by-side panels.
# Left: plot_bloch(ax, part2_res, ..., omega_x=omega1, axis_unit=omega1)
# Right: P0(t) and P1(t) from part2_res, with Time (us) 0-2 and Probability 0-1.
# Add labeled vertical dashed line at theoretical first t_pi = pi/omega1.
# Include title and legend; save fig2 and register figures['Figure 2'] = fig2.

**FIGURE 2 CAPTION —** [Explain how the trajectory corresponds to population transfer]

### CODE — TODO · Double coupling and compare π-pulse durations
**Required output:** two rows (`ω1 = 2π, 4π` rad/µs), columns **ω1, predicted first tπ (µs), simulated first tπ (µs), maximum P1**; report to at least three decimals. Store stronger-drive result as `part2_fast`. No separate figure. Construct your own prompt.

In [ ]:
omega1_fast = 4*np.pi
H_res_fast = nv_hamiltonian(0.0, omega1_fast)
part2_fast = None                 # TODO: simulate_spin(H_res_fast, zero, times2)
# TODO: print 2-row table: omega1 (rad/us), predicted pi-pulse (us),
# simulated first maximum (us), max P1; compare 2*pi and 4*pi.
# Do not add another numbered figure.

## Part 2B — Approaching Resonance
### DO NOW — Predictions
At fixed $\omega_1=2\pi$ rad/$\mu$s, approach the *natural* frequency $\omega_0$ from below using $\Delta\omega/\omega_1=-4,-2,-1,0$. The effective $z$ coefficient is $-\Delta\omega$.

**1. Which path remains closest to the north pole?** [Answer]  
**2. How does maximum $P_1$ change?** [Answer]  
**3. How does the effective axis tilt?** [Answer]  
**4. Do oscillations speed up or slow down?** [Answer]


### CODE — TODO · Four detuning simulations (data for Figure 3)
Use ratios `−4, −2, −1, 0`, constant `omega1=2π` rad/µs, 1,001 time points (0–2 µs), initial `zero`. Reuse `nv_hamiltonian()` and `simulate_spin()`. Store four standard dictionaries in `detuning_results[ratio]`. **No stand-alone plots yet.**

In [ ]:
detuning_ratios = [-4, -2, -1, 0]   # Delta_omega/omega1
# Keys are the SIGNED drive-minus-natural frequency ratios.
detuning_results = {}

for ratio in detuning_ratios:
    detuning_case = ratio * omega1
    # TODO: Build H = nv_hamiltonian(detuning_case, omega1).
    # TODO: Store simulate_spin(H, zero, times2) at detuning_results[ratio].
    pass


### CODE — TODO · Figure 3 (consult Part 2, Step 7 and `Images/figure3_layout_guide.png`)

**One five-panel figure:** four Bloch spheres in a **2×2 grid** ordered `−4, −2, −1, 0` (top-left to bottom-right), above one full-width graph of **four P1(t) curves**. Identical sphere viewing angles and component-arrow scales. Use `omega_z=-ratio*omega1`, `omega_x=omega1`, `axis_unit=omega1`, `axis_scale=0.28` for each sphere. Graph axes Time (µs) 0–2 and P1 0–1; legend, descriptive titles and caption. Register `figures["Figure 3"] = fig3`.

**Prompt construction:** Formulate your own complete LLM prompt, identifying result dictionary access, panel organization, plotting helper arguments, shared scales, legends, and registration.

In [ ]:
# TODO: Figure 3 = ONE Matplotlib figure with a 2x2 Bloch-sphere grid
# (top-left -4, top-right -2, bottom-left -1, bottom-right 0)
# PLUS a full-width P1(t) comparison graph below.
# For each sphere call plot_bloch(ax, detuning_results[ratio], ...,
#     omega_z=-ratio*omega1, omega_x=omega1,
#     axis_unit=omega1, axis_scale=0.28).
# Use identical camera angles, labeled signed detunings, initial/time arrows.
# Bottom graph: 4 P1(t) curves on Time (us) 0-2, P1 0-1 with legend.
# Register figures['Figure 3'] = fig3.

**FIGURE 3 CAPTION —** [Connect drive-minus-natural detuning, effective axis tilt, maximum transfer, and generalized Rabi frequency]


### CODE — TODO · Detuning comparison table
Print rows for signed `Δω/ω1 = −4, −2, −1, 0`, columns **ratio, simulated max P1, theoretical max P1, Ω (rad/µs)**. Calculate the simulated maximum over all sampled times. Three decimals for probabilities. No additional graph.

In [ ]:
# TODO: print 4-row detuning comparison:
# signed ratio | simulated max P1 over all times | theory = 1/(1+ratio**2) |
# generalized Omega = omega1*sqrt(1+ratio**2) (rad/us).
# Expected theoretical maxima: 1/17, 1/5, 1/2, 1.
# Format probability to 3 decimals; no extra graph.

### DO NOW — Independent Physics Analysis
**Q3 — Resonant Spin Control.** Compared with Part 1, how does the resonant Hamiltonian change the rotation axis and allow a complete spin flip with a $\pi$-pulse?

[Type your answer here]

**Q4 — Approaching Resonance and Magnetometry.** Use your four Bloch trajectories and population curves to explain how decreasing detuning changes the rotation axis, maximum transfer, and oscillation frequency. How could a changing magnetic field alter detuning and inform magnetometry?

[Type your answer here]

## Part 3 — Optical Readout and NV Magnetometry

### DO NOW — Predict the fluorescence spectrum
For fixed pulse `tau=0.5` µs, `F0=100`, `F1=70`, and an NV initialized in `zero`, predict fluorescence far from resonance, near resonance and at the central minimum. **Record predictions before the sweep.**

**1. Far from resonance:** [Answer]  
**2. Near resonance:** [Answer]  
**3. Frequency of minimum:** [Answer]  
**4. Which graph feature identifies the NV transition?** [Answer]

Use the existing Part 2 effective Hamiltonian. Distinguish the *natural* frequency `ω0` from the *applied* frequency `ω`, and retain the signed detuning `Δω=ω−ω0`.

### CODE — TODO · Initial 201-point frequency sweep
Fixed microwave pulse `tau=π/omega1=0.5` µs; 201 offsets from −4ω1 to +4ω1. Reuse `nv_hamiltonian()`, `simulate_spin()`, and `fluorescence()`. Store `part3_base` with **four aligned arrays** (`drive_offsets`, `p0`, `p1`, `fluorescence`) and the standard `simulations` list. Report the original central minimum frequency (MHz relative to original) and photon count. A preliminary graph is optional, not numbered.

In [ ]:
tau = np.pi / omega1                  # 0.5 us: fixed for EVERY drive frequency
# Drive angular-frequency offsets relative to the ORIGINAL natural frequency omega0.
# 201 points include zero and the shifted resonance exactly.
drive_offsets = np.linspace(-4*omega1, 4*omega1, 201)


def sweep_fluorescence(drive_offsets, nv_shift=0.0):
    """Sweep fixed-duration microwave pulses using the standard spin solver.

    drive_offsets: omega_drive - ORIGINAL natural omega0 (rad/us)
    nv_shift: magnetic-field-induced change in natural omega0 (rad/us)
    actual detuning = drive_offset - nv_shift = omega_drive - shifted omega0

    Returns the original drive-frequency axis, final populations, fluorescence,
    and each standard simulate_spin() result in 'simulations'.
    """
    # TODO: For each drive_offset:
    #   detuning = drive_offset - nv_shift
    #   H = nv_hamiltonian(detuning, omega1)    # z term is -detuning
    #   sim = simulate_spin(H, zero, [0, tau])
    #   collect sim['p0'][-1] and sim['p1'][-1]
    #   preserve each standard sim dict in 'simulations'
    #   calculate fluorescence(...) from the final populations
    # TODO: Return a dictionary with keys:
    #   'drive_offsets', 'simulations', 'p0', 'p1', 'fluorescence'
    raise NotImplementedError("Complete sweep_fluorescence with existing helpers")

part3_base = None   # TODO: sweep_fluorescence(drive_offsets, nv_shift=0.0)


### DO NOW — Predict the magnetic-field-induced resonance shift
A magnetic-field change shifts the natural transition angular frequency by `nv_shift = omega1`. Keep the **applied drive-frequency axis and pulse duration fixed**.

**1. New resonance higher or lower?** [Prediction]  
**2. Direction of fluorescence-minimum movement?** [Prediction]  
**3. Should resonant minimum fluorescence change?** [Prediction]  
**4. How should the original and shifted spectra compare?** [Prediction]

### CODE — TODO · Recalculate shifted spectrum on the SAME frequency axis
Shifted detuning = `drive_offset - nv_shift`; use the **same `drive_offsets` array**, 201 points and 0.5 µs pulse. Store `part3_shift`; do not overwrite `part3_base`. **Do not make a separate numbered figure.**

In [ ]:
nv_shift = omega1     # imposed increase in natural transition angular frequency
# TODO: Reuse EXACTLY the SAME drive_offsets array.
# The drive-minus-natural detuning is now drive_offset - nv_shift.
part3_shift = None   # TODO: sweep_fluorescence(drive_offsets, nv_shift)


### CODE — TODO · Figure 4: independent scientific graph design
**No layout-guide image is provided for Figure 4.** Generate a single graph comparing BOTH simulated fluorescence curves on the **same** axis of microwave-frequency offset relative to original resonance (MHz, −4 to +4). The y-axis is expected red photons/readout, chosen to reveal both spectra. Clearly distinguish the two curves; mark and label the central minimum of each using measured frequencies; title, legend, units, caption. Register `figures["Figure 4"] = fig4`.

**AI prompt design:** You must independently specify the data arrays, shared x-axis, measured minimum markers and registration. Do not request fabricated curves or a recentered shifted spectrum.

In [ ]:
# TODO: Figure 4. ONE plot with TWO spectra on SAME drive_offsets/(2*pi) axis.
# X: microwave frequency offset relative to ORIGINAL NV resonance (MHz), -4 to +4.
# Y: expected detected red photons per readout window (choose readable limits).
# Plot part3_base['fluorescence'] and part3_shift['fluorescence'].
# Determine and LABEL both CENTRAL minimum frequencies FROM SIMULATED DATA.
# Include descriptive title, legend, distinct curves, units and caption below.
# NO layout image, NO independently recentered frequency axis, NO invented data.
# Register figures['Figure 4'] = fig4.

**FIGURE 4 CAPTION —** [Explain how changing the magnetic field shifts the optical resonance]

### CODE — TODO · Resonance comparison AND verification summary
Print a three-row comparison table (columns **Quantity, Original field, Changed field**): imposed natural-frequency shift (MHz), measured resonance relative to original (MHz), and minimum fluorescence (photons/readout). Then print measured resonance displacement in MHz.

Also print the Part 3 verification table: **maximum normalization error; minimum/maximum photon counts for each spectrum; maximum original-spectrum symmetry discrepancy; measured shift (MHz)**. Use computed values and verify physical limits. The +1 MHz theoretical shift belongs to verification, not data inserted into Figure 4.

In [ ]:
# TODO: print a 3-row comparison table, columns Quantity | Original | Changed:
# imposed natural-frequency shift (MHz), measured central resonance offset (MHz),
# minimum expected fluorescence (photons/readout).
# Print measured shift (shifted minus original resonance) in MHz.
# TODO: print verification SUMMARY with:
# max abs(P0 + P1 - 1) over BOTH sweeps,
# minimum and maximum fluorescence for EACH spectrum,
# maximum symmetry difference for original F(offset) vs F(-offset),
# measured resonance shift in MHz.
# Frequencies in MHz = drive_offsets (rad/us) / (2*pi).
# Calculate minima and errors from actual arrays; don't hardcode expected answers.

### DO NOW — Step 7: Interpret the ODMR resonance shift
**Conceptual question:** Using the measured shift in the NV natural resonance frequency, what physical quantity could you calculate? Explain the relationship between this quantity and the NV spin's energy-level splitting. **No additional coding or numerical calculation is required.**

**Your answer:** [Type your response here]


No additional magnetic-field calculation is requested. Answer independently, without an LLM.

### DO NOW — Independent Physics Analysis
**Q5 — From Spin Dynamics to Fluorescence.** Explain why fluorescence decreases near resonance by connecting detuning, spin-flip probability, and spin-dependent fluorescence yields.

[Type your answer here]

**Q6 — Using NV Centers as Magnetometers.** Explain the chain from magnetic-field change to NV energy splitting to resonance-frequency shift and fluorescence measurement. Why measure the resonance rather than fluorescence at an arbitrary drive frequency?

[Type your answer here]

## AI Development Record — One entry per computational part
*If no AI was used, describe independent development and verification. Do not include full transcripts.*

### AI Development Record 1 — Larmor Precession
**Objective:** [Your response]

**Actual prompt:** [Paste one prompt or write “No AI used”]

**AI contribution:** [Your response]

**Verification:** [Describe your independent numerical / physics check]

**Revision, if applicable:** [Your response]

**Prompt-quality check:** Did your initial AI prompt clearly specify the scientific output? If you revised it to correct a misunderstanding, explain what changed.

### AI Development Record 2 — Microwave Driving and Rabi Oscillations
**Objective:** [Your response]

**Actual prompt:** [Paste one prompt or write “No AI used”]

**AI contribution:** [Your response]

**Verification:** [Describe your independent numerical / physics check]

**Revision, if applicable:** [Your response]

**Prompt-quality check:** Did your initial AI prompt clearly specify the scientific output? If you revised it to correct a misunderstanding, explain what changed.

### AI Development Record 3 — Optical Readout and Magnetometry
**Objective:** [Your response]

**Actual prompt:** [Paste one prompt or write “No AI used”]

**AI contribution:** [Your response]

**Verification:** [Describe your independent numerical / physics check]

**Revision, if applicable:** [Your response]

**Prompt-quality check:** Did your initial AI prompt clearly specify the scientific output? If you revised it to correct a misunderstanding, explain what changed.

### Final AI Reflection — Approximately 100 words
How did AI assistance affect your workflow? Identify one error, limitation, or unexpected result you investigated. Explain one physics insight you gained by interpreting the results rather than merely running code. (If no AI was used, describe independent coding and verification.)

[Type approximately 100 words here]

## Final Submission Check
### CODE — PROVIDED · Run after completing the notebook

In [ ]:
# Incomplete tasks are NOT RUN; completed incorrect results are FAIL.
# This checks numerical output and figure objects, not written work or plot quality.

checks = []

def add_check(label, test, available=True):
    status = 'NOT RUN' if not available else ('PASS' if bool(test) else 'FAIL')
    checks.append((label, status))
    print(f'{status:8} | {label}')

def valid_sim(v):
    return (isinstance(v, dict)
            and all(k in v for k in ('times','states','p0','p1','bloch'))
            and len(v['states']) >= 2)

def valid_sweep(v):
    return (isinstance(v, dict)
            and all(k in v for k in ('drive_offsets','simulations','p0','p1','fluorescence'))
            and len(v['drive_offsets']) == len(v['p1']) == len(v['fluorescence'])
            and len(v['drive_offsets']) >= 100
            and all(valid_sim(sim) for sim in v['simulations'])
            and len(v['simulations']) == len(v['drive_offsets']))

for i in range(1, 5):
    fig = figures.get(f'Figure {i}')
    add_check(f'Figure {i} registered', isinstance(fig, Figure), fig is not None)

if valid_sim(globals().get('part1')):
    norm_err, pop_err = check_normalization(part1)
    add_check('Part 1 state and population normalization', max(norm_err,pop_err) < 1e-5)
    add_check('Part 1 populations constant',
              max(np.ptp(part1['p0']), np.ptp(part1['p1'])) < 1e-5)
else:
    add_check('Part 1 state and population normalization', False, False)
    add_check('Part 1 populations constant', False, False)

if valid_sim(globals().get('part1_fast')):
    add_check('Part 1 doubled-frequency normalization', max(check_normalization(part1_fast)) < 1e-5)
else:
    add_check('Part 1 doubled-frequency normalization', False, False)

if valid_sim(globals().get('part2_res')):
    norm_err, pop_err = check_normalization(part2_res)
    add_check('Resonant Rabi normalization', max(norm_err,pop_err) < 1e-5)
    i_pi = int(np.argmin(np.abs(part2_res['times'] - np.pi/omega1)))
    add_check('Resonant pi-pulse spin flip', part2_res['p1'][i_pi] > 0.995)
else:
    add_check('Resonant Rabi normalization', False, False)
    add_check('Resonant pi-pulse spin flip', False, False)

if valid_sim(globals().get('part2_fast')):
    i_fast = int(np.argmin(np.abs(part2_fast['times'] - np.pi/(2*omega1))))
    add_check('Doubled-coupling pi pulse', part2_fast['p1'][i_fast] > 0.995)
else:
    add_check('Doubled-coupling pi pulse', False, False)

cases = globals().get('detuning_results')
if isinstance(cases, dict) and all(valid_sim(cases.get(r)) for r in [-4,-2,-1,0]):
    add_check('Four detuning simulations normalized',
              all(max(check_normalization(cases[r])) < 1e-5 for r in [-4,-2,-1,0]))
    add_check('Detuning max transfer agrees with theory',
              all(abs(np.max(cases[r]['p1']) - 1/(1+r*r)) < 0.012 for r in [-4,-2,-1,0]))
else:
    add_check('Four detuning simulations normalized', False, False)
    add_check('Detuning max transfer agrees with theory', False, False)

base, shift = globals().get('part3_base'), globals().get('part3_shift')
if valid_sweep(base) and valid_sweep(shift):
    norm_ok = all(max(check_normalization(sim)) < 1e-5
                  for dat in (base,shift) for sim in dat['simulations'])
    add_check('Frequency-sweep populations normalized', norm_ok)
    all_f = np.concatenate([np.asarray(base['fluorescence']), np.asarray(shift['fluorescence'])])
    add_check('Fluorescence within [70, 100]', np.min(all_f)>=70-1e-5 and np.max(all_f)<=100+1e-5)
    same_axis = np.allclose(base['drive_offsets'], shift['drive_offsets'])
    add_check('Both ODMR spectra share drive-frequency axis', same_axis)
    add_check('Unshifted ODMR response symmetric',
              np.allclose(np.asarray(base['fluorescence']), np.asarray(base['fluorescence'])[::-1], atol=0.02))
    # These are central minima because a resonant pi-pulse yields the global fluorescence minimum.
    f0_MHz = base['drive_offsets'][np.argmin(base['fluorescence'])]/(2*np.pi)
    f1_MHz = shift['drive_offsets'][np.argmin(shift['fluorescence'])]/(2*np.pi)
    add_check('Original and shifted central resonances',
              abs(f0_MHz)<0.03 and abs(f1_MHz-1)<0.03)
    add_check('Magnetic-field-induced shift = +1 MHz', abs((f1_MHz-f0_MHz)-1)<0.03)
else:
    for label in ('Frequency-sweep populations normalized','Fluorescence within [70, 100]',
                  'Both ODMR spectra share drive-frequency axis','Unshifted ODMR response symmetric',
                  'Original and shifted central resonances','Magnetic-field-induced shift = +1 MHz'):
        add_check(label, False, False)

from collections import Counter
counts = Counter(status for _,status in checks)
print(f"\nSUMMARY: {counts['PASS']} PASS | {counts['FAIL']} FAIL | {counts['NOT RUN']} NOT RUN")
print('Review captions, Q1–Q6, AI records, reflection, and PDF manually; these cannot be verified here.')

### FINAL DO NOW — PDF submission checklist
- [ ] Analytical warm-up has the four-row phase/population comparison table.
- [ ] Figure 1 has two side-by-side panels; Figure 2 has two panels plus π-pulse marker.
- [ ] Figure 3 has a 2×2 Bloch grid above one common P1(t) graph.
- [ ] Figure 4 contains two fluorescence spectra on a **single original-frequency axis**, both resonance markers, and **no copied example data**.
- [ ] All figures have readable labels, legends, units and captions.
- [ ] The doubled-frequency, doubled-coupling, detuning, resonance, and verification tables are printed.
- [ ] Predictions, Q1–Q6, conceptual magnetometry answer and three AI Development Records are complete.
- [ ] Final ~100-word AI reflection is complete.
- [ ] The provided Submission Check reports appropriate PASS results.
- [ ] Export one PDF, inspect all pages, and submit `NV_Spin_Dynamics_Lastname.pdf` to the LMS.

**The automated Submission Check verifies numerical calculations and the presence of figure objects, but not figure layouts, labels or written responses.**